# Funil de Disparo — Bônus Servidores

Acompanha o ciclo completo dos dois disparos de bônus para servidores municipais:

| Campanha | Template HSM | Tabela fonte |
|---|---|---|
| MobiRio | `recado_mobirio_pref_rio_bonus_v1` | `rj-crm-registry-dev.teste.bonus_mobi_rio` |
| Servidores (2º envio) | `recado_servidores_pref_rio_14` | `rj-crm-registry-dev.teste.bonus_servidor_segundo_envio` |

**Etapas do funil:**
1. Pessoas nas tabelas originais
2. Telefones nulos originalmente
3. Pessoas com todos os telefones invalidados pela `VALIDATE_AND_FORMAT_PHONE`
4. Números efetivamente disparados
5. Erros na plataforma Salesforce
6. Erros na Meta (WhatsApp)
7. Recebidos (OttDelivered confirmado)
8. Lidos (OttRead confirmado)
9. Respondidos
10. Relação da resposta com o HSM enviado (classificação LLM — Agentforce)
11. Detalhamento de falhas

---
**Notas sobre métricas de entrega/leitura:**  
Usamos `entrega_confirmada_indicador` e `leitura_confirmada_indicador` (eventos OttDelivered/OttRead observados diretamente),  
**não** `foi_entregue`/`foi_lida` — essas inferem entrega/leitura a partir de resposta pareada e superestimam (~10pp e ~18pp respectivamente).

In [4]:
import os
import pandas as pd
import pandas_gbq
from google.oauth2 import service_account
from google.cloud import bigquery

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

# ------------------------------------------------------------------
# Credenciais
# As tabelas bonus_mobi_rio e bonus_servidor_segundo_envio são
# tabelas EXTERNAS vinculadas ao Google Sheets. Por isso precisam
# do scope drive.readonly, que o basedosdados não inclui.
# Usamos a service account do dbt com esse scope adicionado.
# A variável GOOGLE_APPLICATION_CREDENTIALS (que aponta para a
# conta do Claude) é ignorada — passamos as credenciais explicitamente.
# ------------------------------------------------------------------
KEYFILE   = os.path.expanduser("~/.dbt/tmp/credentials.json")
SCOPES    = [
    "https://www.googleapis.com/auth/bigquery",
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/drive.readonly",   # necessário para tabelas externas (Sheets)
]
CREDENTIALS   = service_account.Credentials.from_service_account_file(KEYFILE, scopes=SCOPES)
BILLING_PROJECT = "rj-crm-registry"

# cliente BigQuery (para inspeção e queries diretas se necessário)
bq = bigquery.Client(project=BILLING_PROJECT, credentials=CREDENTIALS)

def run_query(sql: str) -> pd.DataFrame:
    """Executa SQL no BigQuery e retorna DataFrame."""
    return pandas_gbq.read_gbq(
        sql,
        project_id=BILLING_PROJECT,
        credentials=CREDENTIALS,
        progress_bar_type=None,
    )

# Nomes dos templates HSM (= campaign_name no prefect.yaml)
HSM_MOBI_RIO   = "recado_mobirio_pref_rio_bonus_v1"
HSM_SERVIDORES = "recado_servidores_pref_rio_14"

# Tabelas BigQuery
TABELA_MOBI_RIO      = "`rj-crm-registry-dev.teste.bonus_mobi_rio`"
TABELA_SERVIDORES    = "`rj-crm-registry-dev.teste.bonus_servidor_segundo_envio`"
FCT_CHATBOT_V2       = "`rj-crm-registry.rmi_conversas.chatbot`"
TABELA_PESSOA_FISICA = "`rj-crm-registry.rmi_dados_mestres.pessoa_fisica`"

# Seed de códigos de falha replicado inline (fonte: seeds/whatsapp_falha_codigo.csv)
# Evita dependência de arquivo externo; keep in sync com o seed do dbt.
SEED_FALHA_SQL = """
    SELECT * FROM UNNEST([
        STRUCT('131026'              AS codigo_sigla, 'meta'       AS origem_tipo, 'Número sem WhatsApp ou sem aceite dos termos atuais'     AS descricao_nome),
        STRUCT('130429',                              'meta',                       'Limite de volume de envio atingido'),
        STRUCT('10',                                  'meta',                       'Permissão da API negada ou removida'),
        STRUCT('131048',                              'meta',                       'Limite de envio por marcação de spam'),
        STRUCT('131000',                              'meta',                       'Erro genérico da Meta'),
        STRUCT('132005',                              'meta',                       'Texto do template longo demais'),
        STRUCT('131049',                              'meta',                       'Não entregue para preservar o engajamento do usuário'),
        STRUCT('131056',                              'meta',                       'Muitas mensagens para o mesmo destinatário em pouco tempo'),
        STRUCT('2',                                   'meta',                       'Indisponibilidade temporária da API'),
        STRUCT('132015',                              'meta',                       'Template pausado por baixa qualidade'),
        STRUCT('130472',                              'meta',                       'Número em experimento da Meta, não enviado'),
        STRUCT('131047',                              'meta',                       'Fora da janela de 24h, exige template'),
        STRUCT('132001',                              'meta',                       'Template inexistente ou não aprovado no idioma'),
        STRUCT('100',                                 'meta',                       'Parâmetro inválido na chamada'),
        STRUCT('FALHA_PROCESSAMENTO', 'plataforma',                                'Falha da plataforma ao processar o envio'),
        STRUCT('ERRO_DESCONHECIDO',   'plataforma',                                'Erro sem código informado pela plataforma'),
        STRUCT('EM_ATENDIMENTO_URA',  'plataforma',                                'Número em atendimento na URA, envio bloqueado')
    ])
"""

print("Setup concluído. Credenciais carregadas de:", KEYFILE)

Setup concluído. Credenciais carregadas de: /home/patricia/.dbt/tmp/credentials.json


## 1. Funil pré-disparo — tabelas originais

Replica a lógica das queries de disparo para medir cada etapa de filtragem:

- **Total de pessoas:** CPFs únicos e não nulos na tabela original
- **Telefones nulos originalmente:** todos os campos de telefone estavam nulos antes de qualquer validação
- **Todos invalidados pela UDF:** tinham algum candidato, mas nenhum passou pela `VALIDATE_AND_FORMAT_PHONE`
- **Com telefone válido:** base final encaminhada ao disparo

In [ ]:
# ------------------------------------------------------------------
# 1a. bonus_mobi_rio
# A query original aplica COALESCE de 4 colunas de telefone,
# cada uma passada pela UDF VALIDATE_AND_FORMAT_PHONE.
# ------------------------------------------------------------------
query_funil_mobi_rio = f"""
WITH raw AS (
    SELECT
        LPAD(CAST(CPFNUMERO AS STRING), 11, '0') AS cpf,
        TELEFONE           AS tel_1,
        TELEFONE_2         AS tel_2,
        Telefone_resgatado AS tel_3,
        TELEFONE_GLOBUS    AS tel_4
    FROM {TABELA_MOBI_RIO}
    WHERE CPFNUMERO IS NOT NULL
),
com_validacao AS (
    SELECT
        cpf,
        -- todos os telefones originais eram nulos?
        (tel_1 IS NULL AND tel_2 IS NULL AND tel_3 IS NULL AND tel_4 IS NULL)
            AS todos_tels_nulos_original,
        -- telefone resultante após a UDF (igual ao COALESCE da query de disparo)
        COALESCE(
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_1),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_2),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_3),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_4)
        ) AS telefone_validado
    FROM raw
)
SELECT
    'MobiRio'                                          AS campanha,
    COUNT(cpf)                                          AS total_pessoas,
    COUNT(DISTINCT cpf)                                 AS total_pessoas_distintas,
    COUNTIF(todos_tels_nulos_original)                  AS telefones_nulos_original,
    COUNTIF(
        NOT todos_tels_nulos_original
        AND telefone_validado IS NULL
    )                                                   AS todos_invalidados_pela_udf,
    COUNTIF(telefone_validado IS NOT NULL)              AS com_telefone_valido
FROM com_validacao
"""

df_funil_mobi_rio = run_query(query_funil_mobi_rio)
df_funil_mobi_rio

,campanha,total_pessoas,total_pessoas_distintas,telefones_nulos_original,todos_invalidados_pela_udf,com_telefone_valido
0,MobiRio,2007,2007,3,43,1961


In [28]:
# ------------------------------------------------------------------
# 1b. bonus_servidor_segundo_envio
# A query original usa 4 candidatos (celular[0/1], TELEFONE[0/1]),
# completa DDD de 9 dígitos pelo RMI ou aplica '21' como fallback,
# depois valida pela UDF e pega o primeiro candidato válido por ordem.
# ------------------------------------------------------------------
query_funil_servidores = f"""
WITH
dados_rmi AS (
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        rmi.telefone.principal.ddd              AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(rmi.telefone.principal.valor, r'\\D', ''), 9)
                                                AS tel_9_digitos_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    WHERE rmi.menor_idade IS FALSE
      AND rmi.obito.indicador IS FALSE
      AND rmi.telefone.principal.estrategia_envio IN ('ENVIAR', 'TESTAR')
      AND rmi.telefone.principal.valor IS NOT NULL

    UNION ALL

    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        t.ddd                                   AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(t.valor, r'\\D', ''), 9)
                                                AS tel_9_digitos_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    LEFT JOIN UNNEST(rmi.telefone.alternativo) AS t
    WHERE rmi.menor_idade IS FALSE
      AND rmi.obito.indicador IS FALSE
      AND t.estrategia_envio IN ('ENVIAR', 'TESTAR')
      AND t.valor IS NOT NULL
),

rmi_agrupado AS (
    SELECT
        cpf,
        ARRAY_AGG(STRUCT(tel_9_digitos_rmi AS num_9, ddd_rmi AS ddd)) AS rmi_tels
    FROM dados_rmi
    GROUP BY cpf
),

servidores_base AS (
    SELECT
        LPAD(CAST(CPF AS STRING), 11, '0') AS cpf,
        -- 4 candidatos de telefone, na ordem de prioridade da query original
        [
            REGEXP_REPLACE(SPLIT(celular,   '/')[SAFE_OFFSET(0)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(celular,   '/')[SAFE_OFFSET(1)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(TELEFONE,  '/')[SAFE_OFFSET(0)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(TELEFONE,  '/')[SAFE_OFFSET(1)], r'\\D', '')
        ] AS candidatos_raw,
        -- todos os campos de origem eram nulos?
        (celular IS NULL AND TELEFONE IS NULL) AS todos_tels_nulos_original
    FROM {TABELA_SERVIDORES}
    WHERE CPF IS NOT NULL
),

candidatos_processados AS (
    SELECT
        s.cpf,
        s.todos_tels_nulos_original,
        cand AS tel_raw,
        idx,
        -- complementa DDD para números de 9 dígitos (RMI ou fallback '21')
        CASE
            WHEN LENGTH(cand) = 9 THEN COALESCE(
                (SELECT CONCAT(r.ddd, cand)
                 FROM UNNEST(rmi.rmi_tels) AS r
                 WHERE r.num_9 = cand
                 LIMIT 1),
                CONCAT('21', cand)
            )
            ELSE cand
        END AS tel_com_ddd
    FROM servidores_base s
    LEFT JOIN rmi_agrupado rmi ON s.cpf = rmi.cpf
    CROSS JOIN UNNEST(s.candidatos_raw) AS cand WITH OFFSET idx
    WHERE cand IS NOT NULL AND cand != ''
),

candidatos_validados AS (
    SELECT
        cpf,
        todos_tels_nulos_original,
        idx,
        `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_com_ddd) AS telefone_validado
    FROM candidatos_processados
),

-- um único telefone por CPF (primeiro candidato válido por ordem)
dados_final AS (
    SELECT
        cpf,
        MAX(todos_tels_nulos_original) AS todos_tels_nulos_original,
        ARRAY_AGG(
            telefone_validado IGNORE NULLS
            ORDER BY idx
            LIMIT 1
        )[SAFE_OFFSET(0)] AS telefone
    FROM candidatos_validados
    GROUP BY cpf
),

-- pessoas cujos 4 candidatos eram string vazia ou nula (sem candidatos a validar)
sem_candidatos AS (
    SELECT DISTINCT cpf
    FROM servidores_base
    WHERE todos_tels_nulos_original
)

SELECT
    'Servidores 2º envio'                                AS campanha,
    (SELECT COUNT(CPF)
         FROM {TABELA_SERVIDORES}
        )                                               AS total_pessoas,
    (SELECT COUNT(DISTINCT CPF)
                 FROM {TABELA_SERVIDORES}
                )                                               AS total_pessoas_distintas,
    (SELECT COUNT(DISTINCT CPF)
     FROM {TABELA_SERVIDORES}
     WHERE CPF IS NOT NULL)                              AS total_cpf_nao_nulo,
    (SELECT COUNT(DISTINCT cpf) FROM sem_candidatos)     AS telefones_nulos_original,
    COUNTIF(
        NOT todos_tels_nulos_original
        AND telefone IS NULL
    )                                                    AS todos_invalidados_pela_udf,
    COUNTIF(telefone IS NOT NULL)                        AS com_telefone_valido
FROM dados_final
"""

df_funil_servidores = run_query(query_funil_servidores)
df_funil_servidores

,campanha,total_pessoas,total_pessoas_distintas,total_cpf_nao_nulo,telefones_nulos_original,todos_invalidados_pela_udf,com_telefone_valido
0,Servidores 2º envio,72985,72836,72836,567,7108,65152


In [7]:
df_pre_disparo = pd.concat([df_funil_mobi_rio, df_funil_servidores], ignore_index=True)
print("=== Funil pré-disparo (tabelas originais) ===")
df_pre_disparo

=== Funil pré-disparo (tabelas originais) ===


,campanha,total_pessoas,telefones_nulos_original,todos_invalidados_pela_udf,com_telefone_valido
0,MobiRio,2007,3,43,1961
1,Servidores 2º envio,72836,567,7108,65152


## 2. Funil de envio — fct_chatbot_v2

Filtra as linhas `fonte = 'HSM'` das duas campanhas e conta telefones únicos em cada etapa.

**Classificação de erros** usa o seed `whatsapp_falha_codigo.csv` (replicado inline):
- `plataforma`: `FALHA_PROCESSAMENTO`, `ERRO_DESCONHECIDO`, `EM_ATENDIMENTO_URA`
- `meta`: códigos 131026, 131048, 131049, etc.

**Entrega e leitura** usam os indicadores confirmados (evento observado diretamente):
- `entrega_confirmada_indicador` = OttDelivered real
- `leitura_confirmada_indicador` = OttRead real

In [9]:
query_funil_envio = f"""
WITH
seed_falha AS (
    {SEED_FALHA_SQL}
),

hsm AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = '{HSM_MOBI_RIO}'   THEN 'MobiRio'
            WHEN hsm.nome_hsm = '{HSM_SERVIDORES}'  THEN 'Servidores 2º envio'
        END AS campanha,
        hsm.id_disparo,
        contato.contato_telefone,
        contato.cpf,
        hsm.indicador_falha,
        hsm.descricao_falha,
        hsm.indicador_falha_meta,
        hsm.falha_meta_descricao,
        hsm.indicador_falha_plataforma,
        hsm.falha_plataforma_descricao,
        hsm.entrega_datahora,
        hsm.envio_datahora,
        hsm.leitura_datahora,
        entrega_observada_indicador,
        leitura_observada_indicador,
        foi_respondida
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND data_particao >= "2026-10-26"
      and mensagem_datahora >= '2026-10-26T10:02:02.109000'
),

-- join com o seed para classificar cada falha por origem
hsm_com_origem AS (
    SELECT
        h.*,
        sf.origem_tipo
    FROM hsm h
    LEFT JOIN seed_falha sf
        ON h.indicador_falha IS TRUE
        AND h.descricao_falha LIKE CONCAT('%', sf.codigo_sigla, '%')
)

SELECT
    campanha,
    COUNT(DISTINCT id_disparo)                                              AS total_disparos,
    COUNT(DISTINCT contato_telefone)                                              AS total_numeros_disparados_dist,
    COUNT(DISTINCT cpf)                                                           AS total_cpfs_disparados_dist,
    -- erros por origem
    COUNT(DISTINCT
        CASE WHEN indicador_falha_plataforma
             THEN contato_telefone END)                                           AS erro_plataforma_salesforce,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
             THEN contato_telefone END)                                           AS erro_meta,
    -- ciclo de entrega (eventos observados, sem inferência por resposta)
    COUNT(DISTINCT
            CASE WHEN envio_datahora IS NOT NULL or entrega_datahora IS NOT NULL 
                    or leitura_datahora IS NOT NULL
                 THEN contato_telefone END)                                         AS enviados,
    COUNT(DISTINCT
            CASE WHEN entrega_datahora IS NOT NULL or leitura_datahora IS NOT NULL
                 THEN contato_telefone END)                                         AS entregues,
    COUNT(DISTINCT
        CASE WHEN entrega_observada_indicador IS TRUE
             THEN contato_telefone END)                                           AS recebidos,
    COUNT(DISTINCT
        CASE WHEN leitura_observada_indicador IS TRUE
             THEN contato_telefone END)                                           AS lidos,
    COUNT(DISTINCT
        CASE WHEN foi_respondida IS TRUE
             THEN contato_telefone END)                                           AS respondidos
FROM hsm_com_origem
GROUP BY campanha
ORDER BY campanha
"""

df_funil_envio = run_query(query_funil_envio)
print("=== Funil de envio (fct_chatbot_v2) ===")
df_funil_envio

=== Funil de envio (fct_chatbot_v2) ===


,campanha,total_disparos,total_numeros_disparados_dist,total_cpfs_disparados_dist,erro_plataforma_salesforce,erro_meta,enviados,entregues,recebidos,lidos,respondidos
0,MobiRio,1962,1915,1962,0,208,1712,1695,1695,1376,17
1,Servidores 2º envio,61312,61058,61312,0,10559,50500,49950,49950,35607,380


In [11]:
erro_meta = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['erro_meta']].values[0][0]
disparados_numeros_distintos = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['total_numeros_disparados_dist']].values[0][0]
enviados_numeros_distintos = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['enviados']].values[0][0]
int(enviados_numeros_distintos+erro_meta)

61059

In [12]:
erro_meta/enviados_numeros_distintos

np.float64(0.20908910891089108)

## 2b. Diagnóstico: decomposição por categorias mutuamente exclusivas (Servidores)

A conta `total_numeros_disparados_dist = erro_meta + erro_plataforma + enviados` não fecha
porque as métricas do funil são `COUNT(DISTINCT contato_telefone)` independentes —
um mesmo número pode ser contado em mais de uma categoria se teve múltiplos eventos
(ex.: erro em uma tentativa e envio em outra).

Esta célula classifica cada telefone em **exatamente uma** categoria, usando as flags
da `fct_chatbot_v2` agregadas por telefone.

In [ ]:
# Diagnóstico: categorias mutuamente exclusivas por contato_telefone
# Campos usados:
#   hsm.indicador_falha_plataforma  — falha ocorreu na plataforma WeTalkie/Salesforce
#   hsm.indicador_falha_meta        — falha retornada pela Meta (código de erro WhatsApp)
#   hsm.envio_meta_datahora         — timestamp do momento em que a Meta aceitou o envio
# Apenas Servidores 2º envio, mesmo filtro de data da query_funil_envio.

query_diagnostico_categorias = f"""
WITH
base AS (
    SELECT
        contato.contato_telefone,
        MAX(hsm.indicador_falha_plataforma)          AS teve_erro_plataforma,
        MAX(hsm.indicador_falha_meta)                AS teve_erro_meta,
        MAX(hsm.envio_meta_datahora IS NOT NULL)     AS teve_envio_meta
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.contato_telefone
),

classificado AS (
    SELECT
        contato_telefone,
        CASE
            WHEN     teve_erro_plataforma AND NOT teve_erro_meta AND NOT teve_envio_meta
                THEN 'erro_plataforma_puro'
            WHEN NOT teve_erro_plataforma AND     teve_erro_meta AND NOT teve_envio_meta
                THEN 'erro_meta_puro'
            WHEN NOT teve_erro_plataforma AND     teve_erro_meta AND     teve_envio_meta
                THEN 'erro_meta_e_enviado'
            WHEN NOT teve_erro_plataforma AND NOT teve_erro_meta AND     teve_envio_meta
                THEN 'enviado_limpo'
            WHEN     teve_erro_plataforma AND     teve_erro_meta
                THEN 'erro_plataforma_e_meta'
            WHEN NOT teve_erro_plataforma AND NOT teve_erro_meta AND NOT teve_envio_meta
                THEN 'sem_flag_nenhuma'
            ELSE 'outro'
        END AS categoria
    FROM base
)

SELECT
    categoria,
    COUNT(DISTINCT contato_telefone) AS qtd_telefones
FROM classificado
GROUP BY categoria
ORDER BY qtd_telefones DESC
"""

df_categorias = run_query(query_diagnostico_categorias)
df_categorias['pct'] = (
    100.0 * df_categorias['qtd_telefones'] / df_categorias['qtd_telefones'].sum()
).round(1).astype(str) + '%'
print(f"Total telefones: {df_categorias['qtd_telefones'].sum():,}")
print()
print("""
Categorias:
  enviado_limpo        — sem erro nenhum, com envio_meta_datahora
  erro_meta_puro       — Meta retornou erro, sem envio_meta_datahora
  erro_plataforma_puro — falha na plataforma, antes de chegar à Meta
  erro_meta_e_enviado  — teve erro Meta E envio_meta (retentativa bem-sucedida ou vice-versa)
  sem_flag_nenhuma     — nem erro nem envio_meta (ciclo incompleto / sem webhook de retorno)
""")
display(df_categorias)

Total telefones: 61,058


Categorias:
  enviado_limpo        — sem erro nenhum, com envio_meta_datahora
  erro_meta_puro       — Meta retornou erro, sem envio_meta_datahora
  erro_plataforma_puro — falha na plataforma, antes de chegar à Meta
  erro_meta_e_enviado  — teve erro Meta E envio_meta (retentativa bem-sucedida ou vice-versa)
  sem_flag_nenhuma     — nem erro nem envio_meta (ciclo incompleto / sem webhook de retorno)



,categoria,qtd_telefones,pct
0,enviado_limpo,50459,82.6%
1,erro_meta_puro,10557,17.3%
2,sem_flag_nenhuma,40,0.1%
3,erro_meta_e_enviado,2,0.0%


### Detalhamento dos 40 `sem_flag_nenhuma`

Os 40 telefones sem nenhuma flag (`envio_meta_datahora`, `indicador_falha_meta`,
`indicador_falha_plataforma` todos nulos) **aparecem na tabela raw
`brutos_salesforce.disparos_efetuados`** — ou seja, foram enviados pelo Salesforce.

Na `fct_chatbot_v2` têm `status_disparo` como `delivered`, `read` ou `sent`,
mas **sem nenhum timestamp de webhook** (nem envio, nem falha).

Hipótese: o Salesforce enviou e a Meta processou (alguns foram até lidos),
mas o **webhook de confirmação de envio não voltou** para a plataforma —
falha de callback. Por isso `envio_meta_datahora = NULL` mesmo para quem
ficou com `status = read`.

In [32]:
# Os 40 telefones sem_flag_nenhuma: sem envio_meta_datahora, sem erro de plataforma,
# sem erro de meta — mas com status_disparo = delivered/read/sent na fct_chatbot_v2.
# Buscando nas tabelas raw para entender o que aconteceu.
#
# Resultado: todos os 40 aparecem em brutos_salesforce.disparos_efetuados
# (foram enviados pelo Salesforce), mas não têm webhook de retorno da Meta
# na fct_chatbot_v2 — falha de callback: a Meta processou mas não confirmou
# o envio de volta para a plataforma.

query_sem_flag_raw = f"""
WITH
sem_flag AS (
    SELECT contato.contato_telefone
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.contato_telefone
    HAVING MAX(hsm.indicador_falha_plataforma) IS NOT TRUE
       AND MAX(hsm.indicador_falha_meta)        IS NOT TRUE
       AND MAX(hsm.envio_meta_datahora IS NOT NULL) IS NOT TRUE
)

SELECT
    d.campanha_nome,
    d.telefone_cidadao,
    d.cpf,
    d.disparo_datahora,
    d.id_disparo_iplan,
    d.extracao_data
FROM `rj-crm-registry.brutos_salesforce.disparos_efetuados` d
WHERE d.campanha_nome = '{HSM_SERVIDORES}'
  AND d.telefone_cidadao IN (SELECT contato_telefone FROM sem_flag)
ORDER BY d.telefone_cidadao, d.disparo_datahora
"""

df_sem_flag_raw = run_query(query_sem_flag_raw)
print(f"Linhas na raw (disparos_efetuados): {len(df_sem_flag_raw)}")
print(f"Telefones distintos: {df_sem_flag_raw['telefone_cidadao'].nunique()}")
display(df_sem_flag_raw)

Linhas na raw (disparos_efetuados): 40
Telefones distintos: 40


,campanha_nome,telefone_cidadao,cpf,disparo_datahora,id_disparo_iplan,extracao_data
0,recado_servidores_pref_rio_14,5521964302649,09900921780,2026-10-26 10:59:05,"2026-10-26 10:59:05|recado_servidores_pref_rio_14|09900921780|{""nome_servidor"": ""Danilo Mattos"", ""LOCALE"": ""BR""}",2026-10-26
1,recado_servidores_pref_rio_14,5521964420162,10382387732,2026-10-26 11:05:08,"2026-10-26 11:05:08|recado_servidores_pref_rio_14|10382387732|{""nome_servidor"": ""Elizabeth Silveira"", ""LOCALE"": ""BR""}",2026-10-26
2,recado_servidores_pref_rio_14,5521964424304,83138900704,2026-10-26 11:29:19,"2026-10-26 11:29:19|recado_servidores_pref_rio_14|83138900704|{""nome_servidor"": ""Jose Filho"", ""LOCALE"": ""BR""}",2026-10-26
3,recado_servidores_pref_rio_14,5521966942608,03762488746,2026-10-26 11:17:14,"2026-10-26 11:17:14|recado_servidores_pref_rio_14|03762488746|{""nome_servidor"": ""Joao Medeiros"", ""LOCALE"": ""BR""}",2026-10-26
4,recado_servidores_pref_rio_14,5521972835286,01243231700,2026-10-26 11:29:19,"2026-10-26 11:29:19|recado_servidores_pref_rio_14|01243231700|{""nome_servidor"": ""Marcia Barboza"", ""LOCALE"": ""BR""}",2026-10-26
5,recado_servidores_pref_rio_14,5521973827617,05435020743,2026-10-26 11:17:14,"2026-10-26 11:17:14|recado_servidores_pref_rio_14|05435020743|{""nome_servidor"": ""Aline Silva"", ""LOCALE"": ""BR""}",2026-10-26
6,recado_servidores_pref_rio_14,5521975162375,08241683723,2026-10-26 10:59:05,"2026-10-26 10:59:05|recado_servidores_pref_rio_14|08241683723|{""nome_servidor"": ""Andre Silva"", ""LOCALE"": ""BR""}",2026-10-26
7,recado_servidores_pref_rio_14,5521975375228,92897908734,2026-10-26 11:11:11,"2026-10-26 11:11:11|recado_servidores_pref_rio_14|92897908734|{""nome_servidor"": ""Paulo Souza"", ""LOCALE"": ""BR""}",2026-10-26
8,recado_servidores_pref_rio_14,5521975431228,08162188703,2026-10-26 11:11:11,"2026-10-26 11:11:11|recado_servidores_pref_rio_14|08162188703|{""nome_servidor"": ""Marielle Botelho"", ""LOCALE"": ""BR""}",2026-10-26
9,recado_servidores_pref_rio_14,5521976742898,04793138730,2026-10-26 11:29:19,"2026-10-26 11:29:19|recado_servidores_pref_rio_14|04793138730|{""nome_servidor"": ""Cristiane Nascimento"", ""LOCALE"": ""BR""}",2026-10-26


In [ ]:
# Busca os 40 telefones sem_flag_nenhuma na tabela sfmc_whatsapp_send_event.
#
# Resultado: todos os 40 aparecem com message_recipient_send_status = 'SENT' —
# o evento de envio chegou no SFMC, mas não foi refletido em envio_meta_datahora
# na fct_chatbot_v2. O mesmo bulk_message_id para todos confirma que é a mesma
# campanha. O problema está no pipeline de ingestão entre o SFMC e a fct.

query_sem_flag_send_event = f"""
WITH
sem_flag AS (
    SELECT contato.contato_telefone
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.contato_telefone
    HAVING MAX(hsm.indicador_falha_plataforma) IS NOT TRUE
       AND MAX(hsm.indicador_falha_meta)        IS NOT TRUE
       AND MAX(hsm.envio_meta_datahora IS NOT NULL) IS NOT TRUE
)

SELECT
    s.mobile_number,
    s.id_contato,
    s.message_recipient_send_status,
    s.event_direction,
    s.engagement_date_time,
    s.bulk_message_id,
    s.data_particao
FROM `rj-crm-registry.brutos_salesforce.sfmc_whatsapp_send_event` s
WHERE s.mobile_number IN (SELECT contato_telefone FROM sem_flag)
  AND s.data_particao >= '2026-10-26'
ORDER BY s.mobile_number, s.engagement_date_time
"""

df_sem_flag_send_event = run_query(query_sem_flag_send_event)
print(f"Linhas: {len(df_sem_flag_send_event)} | Telefones: {df_sem_flag_send_event['mobile_number'].nunique()}")
display(df_sem_flag_send_event)


Linhas: 40 | Telefones: 40


,contato_telefone,cpf,id_disparo,status_disparo,indicador_falha,indicador_falha_plataforma,indicador_falha_meta,criacao_envio_datahora,envio_datahora,envio_meta_datahora,entrega_datahora,entrega_meta_datahora,falha_datahora,falha_meta_datahora,entrega_observada_indicador,leitura_observada_indicador,mensagem_datahora
0,5521964302649,09900921780,0tcw8gw8ajcdbhyc5gt6emaa,delivered,False,False,False,2026-10-26 11:02:44.540,NaT,NaT,2026-10-26 11:02:52.839,2026-10-26 11:02:52.839,NaT,NaT,True,False,2026-10-26 11:02:44.540
1,5521964420162,10382387732,0tcwa1nq9495m11gqvk0sz25,delivered,False,False,False,2026-10-26 11:11:04.745,NaT,NaT,2026-10-26 11:11:10.382,2026-10-26 11:11:10.382,NaT,NaT,True,False,2026-10-26 11:11:04.745
2,5521964424304,83138900704,0tcwexvn9zcng786x7y3fr46,read,False,False,False,2026-10-26 11:37:43.376,NaT,NaT,2026-10-26 11:37:52.219,2026-10-26 11:37:52.219,NaT,NaT,True,True,2026-10-26 11:37:43.376
3,5521966942608,03762488746,0tcwc2ywv98c7a0x8w2q1xtn,read,False,False,False,2026-10-26 11:22:12.628,NaT,NaT,2026-10-26 11:22:23.019,2026-10-26 11:22:23.019,NaT,NaT,True,True,2026-10-26 11:22:12.628
4,5521972835286,01243231700,0tcwf33m85cgnbqcrng7h1yj,delivered,False,False,False,2026-10-26 11:38:37.135,NaT,NaT,2026-10-26 11:38:45.654,2026-10-26 11:38:45.654,NaT,NaT,True,False,2026-10-26 11:38:37.135
5,5521973827617,05435020743,0tcwc9jpd9x7a7nv1aev81yg,read,False,False,False,2026-10-26 11:23:20.350,NaT,NaT,2026-10-26 11:23:29.026,2026-10-26 11:23:29.026,NaT,NaT,True,True,2026-10-26 11:23:20.350
6,5521975162375,08241683723,0tcw8mj8hpp3w1s070btvhzk,read,False,False,False,2026-10-26 11:03:22.542,NaT,NaT,2026-10-26 11:03:32.115,2026-10-26 11:03:32.115,NaT,NaT,True,True,2026-10-26 11:03:22.542
7,5521975375228,92897908734,0tcwarf809dfxvkxyhrg38j9,read,False,False,False,2026-10-26 11:14:57.515,NaT,NaT,2026-10-26 14:00:22.803,2026-10-26 14:00:22.803,NaT,NaT,True,True,2026-10-26 11:14:57.515
8,5521975431228,08162188703,0tcwb5jq07ehyv4em1hwsxbt,read,False,False,False,2026-10-26 11:17:11.756,NaT,NaT,2026-10-26 11:17:18.023,2026-10-26 11:17:18.023,NaT,NaT,True,True,2026-10-26 11:17:11.756
9,5521976742898,04793138730,0tcwevs0s3xka4wgbcrc13vw,delivered,False,False,False,2026-10-26 11:37:22.165,NaT,NaT,2026-10-26 11:37:28.684,2026-10-26 11:37:28.684,NaT,NaT,True,False,2026-10-26 11:37:22.165


In [34]:
# Busca os 40 telefones sem_flag_nenhuma na tabela messaging_events_whatsapp.
#
# Resultado: nenhum dos 40 aparece — o join por contact_point_phone_number
# não bate com o contato_telefone da fct (provável diferença de formato).
# Para a campanha inteira, todos os 61.347 eventos são SEND_STATUS/SENT,
# confirmando que a Meta recebeu os envios — o problema é de mapeamento
# entre o campo de telefone das duas tabelas.

query_sem_flag_messaging_events = f"""
WITH
sem_flag AS (
    SELECT contato.contato_telefone
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.contato_telefone
    HAVING MAX(hsm.indicador_falha_plataforma) IS NOT TRUE
       AND MAX(hsm.indicador_falha_meta)        IS NOT TRUE
       AND MAX(hsm.envio_meta_datahora IS NOT NULL) IS NOT TRUE
)

SELECT
    e.event_type,
    e.send_status,
    e.nome_hsm,
    e.contact_point_phone_number,
    e.error_code,
    e.not_sent_reason,
    e.categoria_meta,
    e.evento_datahora,
    e.data_particao
FROM `rj-crm-registry.brutos_salesforce.messaging_events_whatsapp` e
WHERE e.nome_hsm = '{HSM_SERVIDORES}'
  AND e.contact_point_phone_number IN (SELECT contato_telefone FROM sem_flag)
ORDER BY e.contact_point_phone_number, e.evento_datahora
"""

df_sem_flag_messaging = run_query(query_sem_flag_messaging_events)
print(f"Linhas: {len(df_sem_flag_messaging)} | Telefones: {df_sem_flag_messaging['contact_point_phone_number'].nunique() if len(df_sem_flag_messaging) > 0 else 0}")

# Visão geral de todos os eventos da campanha nessa tabela
query_messaging_resumo = f"""
SELECT event_type, send_status, COUNT(*) AS n
FROM `rj-crm-registry.brutos_salesforce.messaging_events_whatsapp`
WHERE nome_hsm = '{HSM_SERVIDORES}'
GROUP BY event_type, send_status
ORDER BY n DESC
"""
print("\nResumo de eventos na messaging_events_whatsapp para a campanha:")
display(run_query(query_messaging_resumo))

if len(df_sem_flag_messaging) > 0:
    display(df_sem_flag_messaging)


Linhas: 0 | Telefones: 0

Resumo de eventos na messaging_events_whatsapp para a campanha:


,event_type,send_status,n
0,SEND_STATUS,SENT,61347


### Detalhamento dos casos `erro_meta_e_enviado`

Os 2 telefones na categoria `erro_meta_e_enviado` **não são a mesma linha com os dois campos preenchidos**.
São **CPFs diferentes cadastrados com o mesmo número de telefone** na planilha fonte:

- O **primeiro CPF** da fila foi entregue normalmente (`envio_meta_datahora` preenchido, `status = read`)
- Os **CPFs seguintes** com o mesmo número levaram erro `131049` — spam throttle da Meta
  (*muitas mensagens para o mesmo destinatário em pouco tempo*) — com `envio_meta_datahora = NULL`

A categoria `erro_meta_e_enviado` surge porque a classificação agrega por **telefone**:
um telefone entra nela se *algum* CPF teve envio E *outro* CPF teve erro meta.

Isso também explica `total_numeros_disparados_dist < total_cpfs_disparados_dist`:
há CPFs distintos compartilhando o mesmo telefone na planilha.

In [5]:
# Traz todas as linhas da fct_chatbot_v2 para os telefones classificados
# como erro_meta_e_enviado — ou seja, telefones que, agregando por telefone,
# tiveram ao mesmo tempo indicador_falha_meta = TRUE (em algum CPF)
# e envio_meta_datahora IS NOT NULL (em outro CPF).
#
# Importante: NÃO é a mesma linha com os dois campos preenchidos.
# É o mesmo número de telefone cadastrado para CPFs diferentes na planilha:
# o primeiro CPF da fila foi entregue (envio_meta_datahora preenchido);
# os CPFs seguintes levaram erro 131049 (spam throttle da Meta —
# muitas mensagens para o mesmo destinatário em pouco tempo),
# com envio_meta_datahora = NULL.
#
# Isso também explica total_numeros_disparados_dist < total_cpfs_disparados_dist.

query_erro_meta_e_enviado = f"""
WITH
telefones_erro_e_enviado AS (
    SELECT contato.contato_telefone
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.contato_telefone
    HAVING MAX(hsm.indicador_falha_meta) = TRUE
       AND MAX(hsm.envio_meta_datahora IS NOT NULL) = TRUE
)

SELECT
    contato.contato_telefone,
    contato.cpf,
    hsm.id_disparo,
    hsm.status_disparo,
    hsm.indicador_falha_meta,
    hsm.falha_meta_descricao,
    hsm.criacao_envio_datahora,
    hsm.envio_meta_datahora,
    hsm.entrega_meta_datahora,
    hsm.falha_meta_datahora,
    mensagem_datahora
FROM {FCT_CHATBOT_V2}
WHERE fonte = 'HSM'
  AND hsm.nome_hsm = '{HSM_SERVIDORES}'
  AND data_particao >= '2026-10-26'
  AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
  AND contato.contato_telefone IN (SELECT contato_telefone FROM telefones_erro_e_enviado)
ORDER BY contato.contato_telefone, mensagem_datahora
"""

df_erro_meta_e_enviado = run_query(query_erro_meta_e_enviado)
print(f"Linhas: {len(df_erro_meta_e_enviado)}")
print(f"Telefones distintos: {df_erro_meta_e_enviado['contato_telefone'].nunique()}")
print(f"CPFs distintos: {df_erro_meta_e_enviado['cpf'].nunique()}")
display(df_erro_meta_e_enviado)


Linhas: 7
Telefones distintos: 2
CPFs distintos: 7


,contato_telefone,cpf,id_disparo,status_disparo,indicador_falha_meta,falha_meta_descricao,criacao_envio_datahora,envio_meta_datahora,entrega_meta_datahora,falha_meta_datahora,mensagem_datahora
0,5521966752474,14619259756,0tcwc53427w3f77cty9c2f43,read,False,None,2026-10-26 11:22:34.974,2026-10-26 11:22:40.647,2026-10-26 11:23:11.478,NaT,2026-10-26 11:22:34.974
1,5521966752474,01804581755,0tcwcdzzgede517r5skndc4g,failed,True,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",2026-10-26 11:24:05.546,NaT,NaT,2026-10-26 11:24:10.196,2026-10-26 11:24:05.546
2,5521966752474,01861580762,0tcwd2tnzjx1dmp9s7ahd5z1,failed,True,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",2026-10-26 11:27:38.892,NaT,NaT,2026-10-26 11:27:42.232,2026-10-26 11:27:38.892
3,5521966752474,08752365700,0tcwets7v7s5extcbgmqtahc,failed,True,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",2026-10-26 11:37:11.963,NaT,NaT,2026-10-26 11:37:17.435,2026-10-26 11:37:11.963
4,5521966752474,09482011775,0tcweywhwqqthwnxap0xtxe9,failed,True,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",2026-10-26 11:37:53.944,NaT,NaT,2026-10-26 11:37:58.652,2026-10-26 11:37:53.944
5,5521972220186,05724840707,0tcwc48zz0ger4fd86jbae1q,sent,False,None,2026-10-26 11:22:26.067,2026-10-26 11:22:34.046,NaT,NaT,2026-10-26 11:22:26.067
6,5521972220186,09909108739,0tcwcafsk9fsbbdnj6cqmrz6,failed,True,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",2026-10-26 11:23:29.658,NaT,NaT,2026-10-26 11:23:34.370,2026-10-26 11:23:29.658


In [ ]:
erro_meta = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['erro_meta']].values[0][0]
disparados_numeros_distintos = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['total_numeros_disparados_dist']].values[0][0]
enviados_numeros_distintos = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['Enviados']].values[0][0]
falha = df_funil_envio[df_funil_envio["falha_meta_datahora"] == 'Servidores 2º envio'][['Enviados']].values[0][0]
int(enviados_numeros_distintos+erro_meta)

NameError: name 'df_funil_envio' is not defined

In [ ]:
erro_meta = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['erro_meta']].values[0][0]
enviados = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['enviados']].values[0][0]
disparados = df_funil_envio[df_funil_envio["campanha"] == 'Servidores 2º envio'][['total_numeros_disparados_dist']].values[0][0]

print(erro_meta+enviados, "deveria ser igual ao total disparado: ", disparados)

In [1]:
61312-10559  # sent: 50.575 delivery 50.035

50753

### Colunas do funil de envio

| Coluna | O que conta |
|---|---|
| `total_numeros_enviados` | Números de telefone distintos que geraram pelo menos um evento na `chatbot` para o HSM da campanha — ou seja, que efetivamente saíram para disparo na plataforma. |
| `total_cpfs_disparados` | CPFs distintos por trás desses números. Pode ser maior que `total_numeros_enviados` se dois CPFs compartilharam o mesmo telefone. |
| `erro_plataforma_salesforce` | Números em que a falha ocorreu **antes** de chegar à Meta — a plataforma Salesforce/WeTalkie não conseguiu processar o envio (ex.: `FALHA_PROCESSAMENTO`, `EM_ATENDIMENTO_URA`). |
| `erro_meta` | Números em que a Meta retornou um código de erro conhecido (ex.: 131026 — sem WhatsApp, 131048 — spam, 130429 — limite de volume). A lista completa está no seed `whatsapp_falha_codigo.csv`. |
| `erro_sem_classificacao` | Números com status de falha mas cujo código **não está** no seed — pode ser código novo, erro sem código, ou status `PROCESSING` que nunca evoluiu. |
| `recebidos` | Números para os quais o WhatsApp confirmou entrega (`entrega_confirmada_indicador = TRUE`). Baseado no evento `OttDelivered` observado diretamente — **não** no campo `foi_entregue`, que infere entrega a partir de resposta e superestima. |
| `lidos` | Números para os quais o WhatsApp confirmou leitura (`leitura_confirmada_indicador = TRUE`). Baseado no evento `OttRead` — **não** no campo `foi_lida`. |
| `respondidos` | Números que enviaram ao menos uma mensagem de volta após o HSM (qualquer tipo de resposta, sem filtro de conteúdo). |

## 3. Relação da resposta com o HSM — classificação LLM (Agentforce)

A coluna `llm.relacao_hsm_tipo` é preenchida em sessões do Agentforce (`AI_AGENT_*`)  
pela pipeline `rj_crm__agentforce_classificacao_llm` (pode ter até 14 dias de atraso).

O elo entre o HSM e a sessão do agente é o `id_sessao_48h`:
todas as mensagens trocadas nas 48h após o disparo compartilham o mesmo identificador de sessão.

**Valores possíveis de `relacao_hsm_tipo`:**
- `DENTRO_DO_ESCOPO` — conversa sobre o tema do bônus
- `MISTO` — parte relacionada ao bônus, parte fora
- `FORA_DO_ESCOPO` — outro assunto
- `SEM_HSM_ASSOCIADO` — sessão sem HSM associado detectado
- `RESPOSTA_ATRASADA_BTN` — clique de botão tardio, sem demanda real (classificado por regra, sem LLM)

In [ ]:
query_llm = f"""
WITH
-- HSMs das campanhas: pega o id_sessao_48h para ligar às sessões do Agentforce
hsm_campanhas AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = '{HSM_MOBI_RIO}'   THEN 'MobiRio'
            WHEN hsm.nome_hsm = '{HSM_SERVIDORES}'  THEN 'Servidores 2º envio'
        END AS campanha,
        id_sessao_48h,
        contato.contato_telefone
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND id_sessao_48h IS NOT NULL
),

-- classificação LLM das sessões do Agentforce: uma linha por (id_sessao_48h, relacao_hsm_tipo)
ai_agent_classif AS (
    SELECT DISTINCT
        id_sessao_48h,
        llm.relacao_hsm_tipo
    FROM {FCT_CHATBOT_V2}
    WHERE fonte IN ('AI_AGENT_CIDADAO', 'AI_AGENT_AGENTE')
      AND id_sessao_48h IS NOT NULL
      AND llm IS NOT NULL
      AND llm.relacao_hsm_tipo IS NOT NULL
),

-- junta: HSM da campanha ↔ sessão do Agentforce na mesma janela de 48h
joined AS (
    SELECT
        h.campanha,
        h.contato_telefone,
        a.relacao_hsm_tipo
    FROM hsm_campanhas h
    INNER JOIN ai_agent_classif a USING (id_sessao_48h)
)

SELECT
    campanha,
    relacao_hsm_tipo,
    COUNT(DISTINCT contato_telefone) AS total_telefones
FROM joined
GROUP BY campanha, relacao_hsm_tipo
ORDER BY campanha, total_telefones DESC
"""

df_llm = run_query(query_llm)

print("=== Relação da resposta com o HSM (classificação LLM — Agentforce) ===")
if df_llm.empty:                                                                
    
    print()
    print("Nenhuma sessão do Agentforce encontrada para essas campanhas.")
    print("Isso ocorre quando os respondentes interagiram via canal CUSTOMER direto,")
    print("sem passar pelo fluxo do Agentforce, ou a classificação LLM ainda não rodou.")
else:
    display(df_llm)

=== Relação da resposta com o HSM (classificação LLM — Agentforce) ===

Nenhuma sessão do Agentforce encontrada para essas campanhas.
Isso ocorre quando os respondentes interagiram via canal CUSTOMER direto,
sem passar pelo fluxo do Agentforce, ou a classificação LLM ainda não rodou.


In [ ]:
3778+63
            

3841

## 4. Detalhamento das falhas

Todos os códigos/descrições de falha das duas campanhas, com origem classificada
(plataforma Salesforce vs Meta) e contagem de telefones únicos por descrição.

In [ ]:
query_falhas = f"""
WITH
seed_falha AS (
    {SEED_FALHA_SQL}
),              

falhas AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = '{HSM_MOBI_RIO}'   THEN 'MobiRio'
            WHEN hsm.nome_hsm = '{HSM_SERVIDORES}'  THEN 'Servidores 2º envio'
        END AS campanha,
        hsm.descricao_falha,
        contato.contato_telefone
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND hsm.indicador_falha IS TRUE
      and data_particao >= "2026-10-26"
      and mensagem_datahora >= '2026-10-26T10:02:02.109000'
),

falhas_classificadas AS (
    SELECT
        f.campanha,
        COALESCE(sf.origem_tipo,    'não classificado')            AS origem_tipo,
        f.descricao_falha                                          AS descricao_falha,
        COUNT(DISTINCT f.contato_telefone)                         AS total_telefones
    FROM falhas f
    LEFT JOIN seed_falha sf
        ON f.descricao_falha LIKE CONCAT('%', sf.codigo_sigla, '%')
    GROUP BY 1, 2, 3
)

SELECT *
FROM falhas_classificadas
ORDER BY campanha, total_telefones DESC
"""

df_falhas = run_query(query_falhas)
print("=== Detalhamento das falhas ===")
display(df_falhas)

=== Detalhamento das falhas ===


,campanha,origem_tipo,descricao_falha,total_telefones
0,MobiRio,não classificado,Message Undeliverable.,151
1,MobiRio,não classificado,Failed to send message because this user's phone number is part of an experiment,30
2,MobiRio,não classificado,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",27
3,Servidores 2º envio,não classificado,Message Undeliverable.,9017
4,Servidores 2º envio,não classificado,Failed to send message because this user's phone number is part of an experiment,1058
5,Servidores 2º envio,não classificado,"In order to maintain a healthy ecosystem engagement, the message failed to be delivered.",480
6,Servidores 2º envio,não classificado,This message was not delivered to maintain healthy ecosystem engagement.,4


**Message Undeliverable (9017)**
Erro genérico da Meta quando a mensagem não pôde ser entregue. Pode ser por número inválido, conta WhatsApp inexistente, bloqueio do usuário ou falha temporária.

**Failed to send message because this user's phone number is part of an experiment**
A Meta realiza experimentos internos de limitação de entrega — alguns números são colocados em grupos de teste onde mensagens de template são bloqueadas propositalmente. Não há nada que você possa fazer pelo lado do remetente.

**In order to maintain a healthy ecosystem engagement, the message failed to be delivered / This message was not delivered to maintain healthy ecosystem engagement**
São duas variações da mesma política da Meta chamada "ecosystem engagement" — a Meta bloqueia o envio quando detecta que o usuário tem baixo engajamento histórico com mensagens de template (não abre, não responde, ignora). É uma proteção contra spam da plataforma. Quanto mais vezes um usuário ignora mensagens de uma empresa, maior a chance de futuras mensagens serem bloqueadas.

**Resumo prático:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          ** os três últimos erros são bloqueios ativos da Meta por política de engajamento — a mensagem nem chega ao dispositivo do usuário. Não há correção possível pelo lado da Prefeitura; são números que a Meta considera "não receptivos" a mensagens de template.

In [36]:
df_falhas.total_telefones.sum()

np.int64(10770)

### 4a. Falhas por órgão (Servidores 2º envio)

Unidade: CPF (primeiro registro por CPF na planilha, caso haja duplicata).
Percentual calculado sobre o total de CPFs disparados do órgão.

In [35]:
# Falhas por órgão — Servidores 2º envio.
# Unidade de contagem: CPF (COUNT DISTINCT).
# CPF duplicado na planilha: considera apenas a primeira ocorrência (ROW_NUMBER = 1).
# JOIN: fct_chatbot_v2 (flags de falha por CPF) x bonus_servidor_segundo_envio (ORGAO).

query_falhas_por_orgao = f"""
WITH
orgao_por_cpf AS (
    SELECT
        LPAD(CAST(CPF AS STRING), 11, '0') AS cpf,
        ORGAO
    FROM (
        SELECT CPF, ORGAO,
            ROW_NUMBER() OVER (PARTITION BY CPF ORDER BY (SELECT NULL)) AS rn
        FROM {TABELA_SERVIDORES}
        WHERE CPF IS NOT NULL
    )
    WHERE rn = 1
),
fct_por_cpf AS (
    SELECT
        contato.cpf,
        MAX(hsm.indicador_falha)            AS teve_falha,
        MAX(hsm.indicador_falha_meta)       AS teve_erro_meta,
        MAX(hsm.indicador_falha_plataforma) AS teve_erro_plataforma
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm = '{HSM_SERVIDORES}'
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.cpf
)

SELECT
    COALESCE(o.ORGAO, 'SEM_ORGAO')                                           AS orgao,
    COUNT(DISTINCT f.cpf)                                                     AS total_cpfs_disparados,
    COUNT(DISTINCT CASE WHEN f.teve_erro_meta       THEN f.cpf END)          AS erro_meta,
    COUNT(DISTINCT CASE WHEN f.teve_erro_plataforma THEN f.cpf END)          AS erro_plataforma,
    COUNT(DISTINCT CASE WHEN f.teve_falha           THEN f.cpf END)          AS total_falhas,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN f.teve_falha THEN f.cpf END)
        / NULLIF(COUNT(DISTINCT f.cpf), 0), 1)                               AS pct_falha
FROM fct_por_cpf f
LEFT JOIN orgao_por_cpf o ON f.cpf = o.cpf
GROUP BY orgao
ORDER BY total_falhas DESC
"""

df_falhas_por_orgao = run_query(query_falhas_por_orgao)
print(f"Órgãos: {len(df_falhas_por_orgao)}")
display(df_falhas_por_orgao)


Órgãos: 51


,orgao,total_cpfs_disparados,erro_meta,erro_plataforma,total_falhas,pct_falha
0,SME ESCOLAS,24615,3795,0,3795,15.4
1,COMLURB,8251,2647,0,2647,32.1
2,SMS,12193,1910,0,1910,15.7
3,GMRIO,5529,976,0,976,17.7
4,SECONSERVA,719,275,0,275,38.2
5,SME,1933,256,0,256,13.2
6,RIOLUZ,249,104,0,104,41.8
7,RIOSAUDE,613,102,0,102,16.6
8,SMF,572,56,0,56,9.8
9,CVL,720,55,0,55,7.6


### 4a2. Números inválidos por órgão (Servidores 2º envio)

Replica a lógica do funil pré-disparo agrupando por ORGAO.
Unidade: CPF (primeiro registro por CPF na planilha, caso haja duplicata).

- `telefone_nulo`: campos `CELULAR` e `TELEFONE` ambos nulos na planilha
- `invalidados_udf`: tinha candidatos mas nenhum passou na `VALIDATE_AND_FORMAT_PHONE`
- `pct_invalido`: `(telefone_nulo + invalidados_udf) / total_cpfs`

In [36]:
# Números inválidos por órgão — Servidores 2º envio.
# Replica a lógica da query_funil_servidores agrupando por ORGAO.
# CPF duplicado na planilha: considera apenas a primeira ocorrência (ROW_NUMBER = 1).

query_invalidos_por_orgao = f"""
WITH
dados_rmi AS (
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        rmi.telefone.principal.ddd AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(rmi.telefone.principal.valor, r'\\D', ''), 9) AS tel_9_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    WHERE rmi.menor_idade IS FALSE AND rmi.obito.indicador IS FALSE
      AND rmi.telefone.principal.estrategia_envio IN ('ENVIAR', 'TESTAR')
      AND rmi.telefone.principal.valor IS NOT NULL
    UNION ALL
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        t.ddd AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(t.valor, r'\\D', ''), 9) AS tel_9_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    LEFT JOIN UNNEST(rmi.telefone.alternativo) AS t
    WHERE rmi.menor_idade IS FALSE AND rmi.obito.indicador IS FALSE
      AND t.estrategia_envio IN ('ENVIAR', 'TESTAR') AND t.valor IS NOT NULL
),
rmi_agrupado AS (
    SELECT cpf, ARRAY_AGG(STRUCT(tel_9_rmi AS num_9, ddd_rmi AS ddd)) AS rmi_tels
    FROM dados_rmi GROUP BY cpf
),
servidores_base AS (
    SELECT
        LPAD(CAST(CPF AS STRING), 11, '0') AS cpf,
        ORGAO,
        ROW_NUMBER() OVER (PARTITION BY CPF ORDER BY (SELECT NULL)) AS rn,
        [
            REGEXP_REPLACE(SPLIT(CELULAR,  '/')[SAFE_OFFSET(0)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(CELULAR,  '/')[SAFE_OFFSET(1)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(TELEFONE, '/')[SAFE_OFFSET(0)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(TELEFONE, '/')[SAFE_OFFSET(1)], r'\\D', '')
        ] AS candidatos_raw,
        (CELULAR IS NULL AND TELEFONE IS NULL) AS todos_tels_nulos_original
    FROM {TABELA_SERVIDORES}
    WHERE CPF IS NOT NULL
),
servidores_dedup AS (
    SELECT cpf, ORGAO, candidatos_raw, todos_tels_nulos_original
    FROM servidores_base WHERE rn = 1
),
candidatos_processados AS (
    SELECT s.cpf, s.ORGAO, s.todos_tels_nulos_original, cand, idx,
        CASE
            WHEN LENGTH(cand) = 9 THEN COALESCE(
                (SELECT CONCAT(r.ddd, cand) FROM UNNEST(rmi.rmi_tels) AS r WHERE r.num_9 = cand LIMIT 1),
                CONCAT('21', cand)
            )
            ELSE cand
        END AS tel_com_ddd
    FROM servidores_dedup s
    LEFT JOIN rmi_agrupado rmi ON s.cpf = rmi.cpf
    CROSS JOIN UNNEST(s.candidatos_raw) AS cand WITH OFFSET idx
    WHERE cand IS NOT NULL AND cand != ''
),
candidatos_validados AS (
    SELECT cpf, ORGAO, todos_tels_nulos_original, idx,
        `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_com_ddd) AS tel_valido
    FROM candidatos_processados
),
dados_final AS (
    SELECT cpf, ORGAO,
        MAX(todos_tels_nulos_original) AS todos_tels_nulos_original,
        ARRAY_AGG(tel_valido IGNORE NULLS ORDER BY idx LIMIT 1)[SAFE_OFFSET(0)] AS tel_valido
    FROM candidatos_validados
    GROUP BY cpf, ORGAO
),
sem_candidatos AS (
    SELECT DISTINCT cpf FROM servidores_dedup WHERE todos_tels_nulos_original
),
todos_cpfs AS (
    SELECT cpf, ORGAO FROM servidores_dedup
)

SELECT
    COALESCE(t.ORGAO, 'SEM_ORGAO')                                           AS orgao,
    COUNT(DISTINCT t.cpf)                                                     AS total_cpfs,
    COUNT(DISTINCT CASE WHEN sc.cpf IS NOT NULL THEN t.cpf END)              AS telefone_nulo,
    COUNT(DISTINCT CASE WHEN df.todos_tels_nulos_original IS FALSE
                         AND df.tel_valido IS NULL THEN t.cpf END)           AS invalidados_udf,
    COUNT(DISTINCT CASE WHEN df.tel_valido IS NOT NULL THEN t.cpf END)       AS validos,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN df.tel_valido IS NULL THEN t.cpf END)
        / NULLIF(COUNT(DISTINCT t.cpf), 0), 1)                               AS pct_invalido
FROM todos_cpfs t
LEFT JOIN sem_candidatos sc ON t.cpf = sc.cpf
LEFT JOIN dados_final df ON t.cpf = df.cpf
GROUP BY orgao
ORDER BY pct_invalido DESC
"""

df_invalidos_por_orgao = run_query(query_invalidos_por_orgao)
print(f"Órgãos: {len(df_invalidos_por_orgao)}")
display(df_invalidos_por_orgao)


Órgãos: 52


,orgao,total_cpfs,telefone_nulo,invalidados_udf,validos,pct_invalido
0,RIOTUR,170,136,1,33,80.6
1,SMFP,2,0,1,1,50.0
2,COMLURB,14565,110,6188,8259,43.3
3,CETRIO,305,36,16,253,17.0
4,FCA,20,2,1,17,15.0
5,MULTIRIO,114,3,8,103,9.6
6,SMCT,23,2,0,21,8.7
7,RIOFILME,28,2,0,26,7.1
8,SMC,143,5,2,136,4.9
9,SMTR,236,6,5,225,4.7


## 4b. Telefones bloqueados pela `get_failed_phones` (quarentena pré-disparo)

O flow do Prefect chama `get_failed_phones` **antes** de disparar, removendo qualquer telefone
que tenha tido falha de quarentena (códigos `131026` ou `131048`) em qualquer campanha
nos últimos 6 meses. Por isso esses telefones nunca aparecem no `fct_chatbot_v2` —
o cruzamento precisa ser feito contra as tabelas originais (com telefone válido após a UDF).

**Fontes consultadas pela função (sem filtro de campanha):**
- WeTalkie (`brutos_wetalkie_staging.fluxo_atendimento_*`): telefone cujo **último** disparo
  nos últimos 6 meses terminou com status FAILED + código 131026 ou 131048
- Salesforce (`brutos_salesforce.status_disparo`): `indicador_quarentena = TRUE`
  nos últimos 6 meses (cobre qualquer campanha)

In [9]:
# Replica exatamente a get_failed_phones(campaign_name=None) do flow
# e cruza com a base de telefones válidos de cada campanha.
# Data de referência: 2026-10-25 (dia do disparo).

DATA_DISPARO = "2026-10-25"

FAILED_PHONES_CTE = f"""
    status_por_disparo AS (
        SELECT
            flatTarget,
            createDate,
            MAX(CASE
                WHEN status = 'PROCESSING' THEN 1
                WHEN status = 'FAILED'
                     AND (faultdescription LIKE '%131026%' OR faultdescription LIKE '%131048%')
                THEN 2
                WHEN status = 'SENT'      THEN 3
                WHEN status = 'DELIVERED' THEN 4
                WHEN status = 'READ'      THEN 5
            END) AS id_status_disparo
        FROM `rj-crm-registry.brutos_wetalkie_staging.fluxo_atendimento_*`
        WHERE DATE(createDate) >= DATE_SUB(DATE '{DATA_DISPARO}', INTERVAL 6 MONTH)
        GROUP BY flatTarget, createDate
    ),
    ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY flatTarget ORDER BY createDate DESC) AS rn
        FROM status_por_disparo
    ),
    wetalkie_failed AS (
        SELECT flatTarget AS telefone
        FROM ranked
        WHERE rn = 1 AND id_status_disparo = 2
    ),
    sf_failed AS (
        SELECT DISTINCT contato_telefone AS telefone
        FROM `rj-crm-registry.brutos_salesforce.status_disparo`
        WHERE indicador_quarentena = TRUE
          AND data_particao >= DATE_SUB(DATE '{DATA_DISPARO}', INTERVAL 6 MONTH)
    ),
    failed_phones AS (
        SELECT telefone FROM wetalkie_failed
        UNION DISTINCT
        SELECT telefone FROM sf_failed
    )
"""

# --- MobiRio ---
query_quarentena_mobi = f"""
WITH
{FAILED_PHONES_CTE},
mobi_raw AS (
    SELECT
        LPAD(CAST(CPFNUMERO AS STRING), 11, '0') AS cpf,
        COALESCE(
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(TELEFONE),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(TELEFONE_2),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(Telefone_resgatado),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(TELEFONE_GLOBUS)
        ) AS tel_validado
    FROM {TABELA_MOBI_RIO}
    WHERE CPFNUMERO IS NOT NULL
),
mobi_validos AS (
    SELECT cpf, tel_validado FROM mobi_raw WHERE tel_validado IS NOT NULL
    UNION ALL SELECT '00000000353', '5511974243228'
)
SELECT
    'MobiRio' AS campanha,
    COUNT(DISTINCT m.tel_validado)                                               AS base_com_telefone_valido,
    COUNT(DISTINCT CASE WHEN fp.telefone IS NOT NULL THEN m.tel_validado END)    AS bloqueados_quarentena
FROM mobi_validos m
LEFT JOIN failed_phones fp ON fp.telefone = m.tel_validado
"""

# --- Servidores 2º envio ---
query_quarentena_serv = f"""
WITH
{FAILED_PHONES_CTE},
dados_rmi AS (
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        rmi.telefone.principal.ddd AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(rmi.telefone.principal.valor, r'\\D', ''), 9) AS tel_9_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    WHERE rmi.menor_idade IS FALSE AND rmi.obito.indicador IS FALSE
      AND rmi.telefone.principal.estrategia_envio IN ('ENVIAR', 'TESTAR')
      AND rmi.telefone.principal.valor IS NOT NULL
    UNION ALL
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        t.ddd AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(t.valor, r'\\D', ''), 9) AS tel_9_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    LEFT JOIN UNNEST(rmi.telefone.alternativo) AS t
    WHERE rmi.menor_idade IS FALSE AND rmi.obito.indicador IS FALSE
      AND t.estrategia_envio IN ('ENVIAR', 'TESTAR') AND t.valor IS NOT NULL
),
rmi_agrupado AS (
    SELECT cpf, ARRAY_AGG(STRUCT(tel_9_rmi AS num_9, ddd_rmi AS ddd)) AS rmi_tels
    FROM dados_rmi GROUP BY cpf
),
servidores_base AS (
    SELECT
        LPAD(CAST(CPF AS STRING), 11, '0') AS cpf,
        [
            REGEXP_REPLACE(SPLIT(celular,  '/')[SAFE_OFFSET(0)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(celular,  '/')[SAFE_OFFSET(1)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(TELEFONE, '/')[SAFE_OFFSET(0)], r'\\D', ''),
            REGEXP_REPLACE(SPLIT(TELEFONE, '/')[SAFE_OFFSET(1)], r'\\D', '')
        ] AS candidatos_raw
    FROM {TABELA_SERVIDORES}
    WHERE CPF IS NOT NULL
),
candidatos_processados AS (
    SELECT s.cpf, cand AS tel_raw, idx,
        CASE
            WHEN LENGTH(cand) = 9 THEN COALESCE(
                (SELECT CONCAT(r.ddd, cand) FROM UNNEST(rmi.rmi_tels) AS r WHERE r.num_9 = cand LIMIT 1),
                CONCAT('21', cand)
            )
            ELSE cand
        END AS tel_com_ddd
    FROM servidores_base s
    LEFT JOIN rmi_agrupado rmi ON s.cpf = rmi.cpf
    CROSS JOIN UNNEST(s.candidatos_raw) AS cand WITH OFFSET idx
    WHERE cand IS NOT NULL AND cand != ''
),
candidatos_validados AS (
    SELECT cpf, idx,
        `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_com_ddd) AS tel_validado
    FROM candidatos_processados
),
servidores_primeiro_valido AS (
    SELECT cpf, tel_validado
    FROM (
        SELECT cpf, tel_validado,
            ROW_NUMBER() OVER (PARTITION BY cpf ORDER BY idx) AS rn_tel
        FROM candidatos_validados
        WHERE tel_validado IS NOT NULL
    )
    WHERE rn_tel = 1
    UNION ALL SELECT '00000000354', '5511974243228'
)
SELECT
    'Servidores 2º envio' AS campanha,
    COUNT(DISTINCT sv.tel_validado)                                               AS base_com_telefone_valido,
    COUNT(DISTINCT CASE WHEN fp.telefone IS NOT NULL THEN sv.tel_validado END)    AS bloqueados_quarentena
FROM servidores_primeiro_valido sv
LEFT JOIN failed_phones fp ON fp.telefone = sv.tel_validado
"""

print("Rodando MobiRio...")
df_q_mobi = run_query(query_quarentena_mobi)
print("Rodando Servidores...")
df_q_serv = run_query(query_quarentena_serv)

df_quarentena = pd.concat([df_q_mobi, df_q_serv], ignore_index=True)
df_quarentena['pct_bloqueados'] = (
    100.0 * df_quarentena['bloqueados_quarentena'] / df_quarentena['base_com_telefone_valido']
).round(1).astype(str) + '%'

print("\n=== Telefones bloqueados pela get_failed_phones (quarentena pré-disparo) ===")
print("(Janela: 6 meses anteriores a 2026-10-25, qualquer campanha)")
display(df_quarentena)

Rodando MobiRio...
Rodando Servidores...

=== Telefones bloqueados pela get_failed_phones (quarentena pré-disparo) ===
(Janela: 6 meses anteriores a 2026-10-25, qualquer campanha)


,campanha,base_com_telefone_valido,bloqueados_quarentena,pct_bloqueados
0,MobiRio,1915,0,0.0%
1,Servidores 2º envio,64836,63,0.1%


## 5. Funil consolidado

Todas as etapas em uma tabela única, com percentual em relação ao total de pessoas na tabela original.

In [10]:
funis = []

for _, row_pre in df_pre_disparo.iterrows():
    campanha = row_pre['campanha']
    total    = int(row_pre['total_pessoas'])

    row_env  = df_funil_envio[df_funil_envio['campanha'] == campanha]
    env = row_env.iloc[0] if not row_env.empty else None

    def pct(val):
        return round(100 * int(val) / total, 1) if total > 0 else None

    row_quar = df_quarentena[df_quarentena['campanha'] == campanha]
    quar = row_quar.iloc[0] if not row_quar.empty else None

    etapas = [
        ('1. Total na tabela original',                 row_pre['total_pessoas']),
        ('2.   └─ Telefones nulos originalmente',       row_pre['telefones_nulos_original']),
        ('3.   └─ Todos invalidados pela UDF',          row_pre['todos_invalidados_pela_udf']),
        ('4.   └─ Com telefone válido (base disparo)',  row_pre['com_telefone_valido']),
    ]

    if quar is not None:
        etapas += [
            ('4b.  └─ Bloqueados por quarentena (get_failed_phones)', quar['bloqueados_quarentena']),
        ]

    if env is not None:
        etapas += [
            ('5. Números disparados (webhook recebido)',    env['total_numeros_enviados']),
            ('6.   └─ Erro plataforma Salesforce',         env['erro_plataforma_salesforce']),
            ('7.   └─ Erro Meta (WhatsApp)',                env['erro_meta']),
            ('8.   └─ Erro sem classificação',             env['erro_sem_classificacao']),
            ('9. Recebidos (OttDelivered confirmado)',      env['recebidos_ott_delivered']),
            ('10. Lidos (OttRead confirmado)',              env['lidos_ott_read']),
            ('11. Respondidos',                            env['respondidos']),
        ]

    for etapa, valor in etapas:
        funis.append({
            'campanha':    campanha,
            'etapa':       etapa,
            'quantidade':  int(valor),
            'pct_do_total': f"{pct(valor)}%",
        })

df_consolidado = pd.DataFrame(funis)

for campanha in df_consolidado['campanha'].unique():
    print(f"\n{'='*65}")
    print(f"  {campanha}")
    print(f"{'='*65}")
    subset = df_consolidado[
        df_consolidado['campanha'] == campanha
    ][['etapa', 'quantidade', 'pct_do_total']].reset_index(drop=True)
    display(subset)

# Classificação LLM (se houver)
if not df_llm.empty:
    print("\n=== Relação das respostas com o HSM (Agentforce LLM) ===")
    display(df_llm)
else:
    print("\nClassificação LLM: sem sessões do Agentforce para essas campanhas.")

NameError: name 'df_pre_disparo' is not defined

## 6. CSV individual por pessoa

Uma linha por CPF com nome, órgão, telefones originais da planilha,
telefone selecionado pela UDF, telefone efetivamente disparado e status final.

**Status final** (prioridade decrescente):
`lido` > `entregue` > `enviado` > `falha_meta` > `falha_plataforma` >
`sem_retorno_webhook` > `invalidado_udf` > `telefone_nulo_original` > `nao_disparado`

O arquivo é salvo na mesma pasta do notebook com separador `;` e encoding `utf-8-sig`
(compatível com Excel Brasil sem precisar de importação manual).
Colunas de telefone são prefixadas com `'` para o Excel tratar como texto.

In [ ]:
query_status_por_pessoa = f"""
WITH
-- MobiRio: tabela original deduplicada por CPF
mobi_raw AS (
    SELECT
        LPAD(CAST(CPFNUMERO AS STRING), 11, '0') AS cpf,
        NOMECOMPLETOFUNC                          AS nome,
        SIGLA                                     AS orgao,
        TELEFONE                                  AS tel_orig_1,
        TELEFONE_2                                AS tel_orig_2,
        Telefone_resgatado                        AS tel_orig_3,
        TELEFONE_GLOBUS                           AS tel_orig_4,
        COALESCE(
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(TELEFONE),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(TELEFONE_2),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(Telefone_resgatado),
            `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(TELEFONE_GLOBUS)
        )                                         AS tel_validado,
        ROW_NUMBER() OVER (PARTITION BY CPFNUMERO ORDER BY (SELECT NULL)) AS rn
    FROM {TABELA_MOBI_RIO}
    WHERE CPFNUMERO IS NOT NULL
),
mobi_dedup AS (
    SELECT cpf, nome, 'MobiRio' AS campanha, orgao,
        tel_orig_1, tel_orig_2, tel_orig_3, tel_orig_4, tel_validado
    FROM mobi_raw WHERE rn = 1
),

-- Servidores: tabela original deduplicada por CPF
serv_raw AS (
    SELECT
        LPAD(CAST(CPF AS STRING), 11, '0') AS cpf,
        NOME                               AS nome,
        ORGAO                              AS orgao,
        CELULAR                            AS tel_orig_1,
        TELEFONE                           AS tel_orig_2,
        CAST(NULL AS STRING)               AS tel_orig_3,
        CAST(NULL AS STRING)               AS tel_orig_4,
        ROW_NUMBER() OVER (PARTITION BY CPF ORDER BY (SELECT NULL)) AS rn
    FROM {TABELA_SERVIDORES}
    WHERE CPF IS NOT NULL
),
serv_dedup AS (
    SELECT cpf, nome, 'Servidores 2º envio' AS campanha, orgao,
        tel_orig_1, tel_orig_2, tel_orig_3, tel_orig_4
    FROM serv_raw WHERE rn = 1
),

-- Servidores: telefone validado via UDF + lookup RMI
dados_rmi AS (
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        rmi.telefone.principal.ddd AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(rmi.telefone.principal.valor, r'\\D', ''), 9) AS tel_9_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    WHERE rmi.menor_idade IS FALSE AND rmi.obito.indicador IS FALSE
      AND rmi.telefone.principal.estrategia_envio IN ('ENVIAR', 'TESTAR')
      AND rmi.telefone.principal.valor IS NOT NULL
    UNION ALL
    SELECT
        LPAD(CAST(rmi.cpf AS STRING), 11, '0') AS cpf,
        t.ddd AS ddd_rmi,
        RIGHT(REGEXP_REPLACE(t.valor, r'\\D', ''), 9) AS tel_9_rmi
    FROM {TABELA_PESSOA_FISICA} AS rmi
    LEFT JOIN UNNEST(rmi.telefone.alternativo) AS t
    WHERE rmi.menor_idade IS FALSE AND rmi.obito.indicador IS FALSE
      AND t.estrategia_envio IN ('ENVIAR', 'TESTAR') AND t.valor IS NOT NULL
),
rmi_agrupado AS (
    SELECT cpf, ARRAY_AGG(STRUCT(tel_9_rmi AS num_9, ddd_rmi AS ddd)) AS rmi_tels
    FROM dados_rmi GROUP BY cpf
),
serv_candidatos AS (
    SELECT s.cpf, cand, idx,
        CASE
            WHEN LENGTH(cand) = 9 THEN COALESCE(
                (SELECT CONCAT(r.ddd, cand) FROM UNNEST(rmi.rmi_tels) AS r WHERE r.num_9 = cand LIMIT 1),
                CONCAT('21', cand)
            )
            ELSE cand
        END AS tel_com_ddd
    FROM (
        SELECT cpf,
            REGEXP_REPLACE(SPLIT(tel_orig_1, '/')[SAFE_OFFSET(0)], r'\\D', '') AS c0,
            REGEXP_REPLACE(SPLIT(tel_orig_1, '/')[SAFE_OFFSET(1)], r'\\D', '') AS c1,
            REGEXP_REPLACE(SPLIT(tel_orig_2, '/')[SAFE_OFFSET(0)], r'\\D', '') AS c2,
            REGEXP_REPLACE(SPLIT(tel_orig_2, '/')[SAFE_OFFSET(1)], r'\\D', '') AS c3
        FROM serv_dedup
    ) s
    LEFT JOIN rmi_agrupado rmi ON s.cpf = rmi.cpf
    CROSS JOIN UNNEST([s.c0, s.c1, s.c2, s.c3]) AS cand WITH OFFSET idx
    WHERE cand IS NOT NULL AND cand != ''
),
serv_validados AS (
    SELECT cpf, idx,
        `rj-crm-registry.udf.VALIDATE_AND_FORMAT_PHONE`(tel_com_ddd) AS tel_validado
    FROM serv_candidatos
),
serv_primeiro_valido AS (
    SELECT cpf,
        ARRAY_AGG(tel_validado IGNORE NULLS ORDER BY idx LIMIT 1)[SAFE_OFFSET(0)] AS tel_validado
    FROM serv_validados GROUP BY cpf
),

-- Base unificada
base AS (
    SELECT cpf, nome, campanha, orgao,
        tel_orig_1, tel_orig_2, tel_orig_3, tel_orig_4, tel_validado
    FROM mobi_dedup
    UNION ALL
    SELECT s.cpf, s.nome, s.campanha, s.orgao,
        s.tel_orig_1, s.tel_orig_2, s.tel_orig_3, s.tel_orig_4,
        sv.tel_validado
    FROM serv_dedup s
    LEFT JOIN serv_primeiro_valido sv ON s.cpf = sv.cpf
),

-- Status final por CPF na fct_chatbot_v2
fct_por_cpf AS (
    SELECT
        contato.cpf,
        contato.contato_telefone                 AS tel_disparado,
        MAX(hsm.indicador_falha_meta)            AS teve_erro_meta,
        MAX(hsm.indicador_falha_plataforma)      AS teve_erro_plataforma,
        MAX(hsm.envio_meta_datahora IS NOT NULL) AS teve_envio_meta,
        MAX(entrega_observada_indicador)         AS foi_entregue,
        MAX(leitura_observada_indicador)         AS foi_lido
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('{HSM_MOBI_RIO}', '{HSM_SERVIDORES}')
      AND data_particao >= '2026-10-26'
      AND mensagem_datahora >= '2026-10-26T10:02:02.109000'
    GROUP BY contato.cpf, contato.contato_telefone
),
fct_status AS (
    SELECT cpf, tel_disparado,
        CASE
            WHEN foi_lido                                THEN 'lido'
            WHEN foi_entregue                            THEN 'entregue'
            WHEN teve_envio_meta AND NOT teve_erro_meta  THEN 'enviado'
            WHEN teve_erro_meta                          THEN 'falha_meta'
            WHEN teve_erro_plataforma                    THEN 'falha_plataforma'
            ELSE                                              'sem_retorno_webhook'
        END AS status_final
    FROM fct_por_cpf
)

SELECT
    b.campanha,
    b.cpf,
    b.nome,
    b.orgao,
    b.tel_orig_1,
    b.tel_orig_2,
    b.tel_orig_3,
    b.tel_orig_4,
    b.tel_validado  AS tel_selecionado_udf,
    f.tel_disparado,
    CASE
        WHEN b.tel_validado IS NULL
             AND b.tel_orig_1 IS NULL
             AND b.tel_orig_2 IS NULL
             AND b.tel_orig_3 IS NULL
             AND b.tel_orig_4 IS NULL  THEN 'telefone_nulo_original'
        WHEN b.tel_validado IS NULL    THEN 'invalidado_udf'
        ELSE COALESCE(f.status_final, 'nao_disparado')
    END             AS status_final
FROM base b
LEFT JOIN fct_status f ON b.cpf = f.cpf
ORDER BY b.campanha, b.cpf
"""

import os

print("Rodando query...")
df_status_por_pessoa = run_query(query_status_por_pessoa)
print(f"Linhas: {len(df_status_por_pessoa)} | CPFs: {df_status_por_pessoa['cpf'].nunique()}")
print()
print(df_status_por_pessoa['status_final'].value_counts())

# Gera CSV compatível com Excel Brasil:
# - separador ; (Excel abre sem precisar configurar importação)
# - encoding utf-8-sig (BOM — Excel reconhece acentos)
# - colunas de telefone prefixadas com ' (Excel trata como texto, não converte para número)
TEL_COLS = ['tel_orig_1', 'tel_orig_2', 'tel_orig_3', 'tel_orig_4',
            'tel_selecionado_udf', 'tel_disparado']

df_excel = df_status_por_pessoa.copy().astype(str)
df_excel = df_excel.replace({'nan': '', 'None': '', '<NA>': ''})

for col in TEL_COLS:
    df_excel[col] = df_excel[col].apply(
        lambda v: f"'{v}" if v.strip() not in ('', 'nan', 'None') else ''
    )

OUTPUT_CSV = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'status_por_pessoa.csv')
# salva na mesma pasta do notebook
OUTPUT_CSV = os.path.join(
    '/home/patricia/Documentos/Work/escritorio_dados/prefeitura-rio'
    '/prefect_rj_iplanrio/pipelines/rj_crm__disparo_template/queries/sma/bonus_servidores',
    'status_por_pessoa.csv'
)

df_excel.to_csv(OUTPUT_CSV, index=False, sep=';', encoding='utf-8-sig')
print(f"\nCSV salvo em: {OUTPUT_CSV}")
print(f"Tamanho: {os.path.getsize(OUTPUT_CSV) / 1024 / 1024:.1f} MB")


Rodando query...
Linhas: 74843 | CPFs: 74843

status_final
lido                      37065
entregue                  14690
falha_meta                10948
invalidado_udf             7160
nao_disparado              3841
telefone_nulo_original      570
enviado                     568
sem_retorno_webhook           1
Name: count, dtype: int64

CSV salvo em: /home/patricia/Documentos/Work/escritorio_dados/prefeitura-rio/prefect_rj_iplanrio/pipelines/rj_crm__disparo_template/queries/sma/bonus_servidores/status_por_pessoa.csv
Tamanho: 9.1 MB


In [38]:
df_status_por_pessoa.head(
    
)

,campanha,cpf,nome,orgao,tel_orig_1,tel_orig_2,tel_orig_3,tel_orig_4,tel_selecionado_udf,tel_disparado,status_final
0,MobiRio,00012609790,JOBER GAMA DE MATOS,E,21982991675,None,#N/A,None,5521982991675,5521982991675,lido
1,MobiRio,00016104498,IVANILDO VALENTIM DA COSTA,E,21980114190,21982857865,#N/A,21979672220,5521980114190,5521980114190,lido
2,MobiRio,00041230779,MOISES SILVA PORTELA,E,21 979023807,19996400790,#N/A,21997604079,5521979023807,5521979023807,lido
3,MobiRio,00041465733,ADERNOEL SANTOS DE MENESES,E,21975921465,21992313619,#N/A,21983236799,5521975921465,5521975921465,lido
4,MobiRio,00056992742,PAULO CEZAR TIMOTEO DA SILVA,E,21 967202303,21/972209655,#N/A,21/969537658,5521967202303,5521967202303,lido


In [39]:
df_status_por_pessoa.status_final.unique()

array(['lido', 'invalidado_udf', 'falha_meta', 'entregue', 'enviado',
       'telefone_nulo_original', 'nao_disparado', 'sem_retorno_webhook'],
      dtype=object)

---
## Pendências de investigação

### 131049 — spam throttle da Meta (Servidores 2º envio)

A `df_falhas` registrou **480 telefones** com o erro `131049`
(*Não entregue para preservar o engajamento do usuário*).

Na análise de `erro_meta_e_enviado` identificamos que esse erro ocorre quando
o mesmo número de telefone está cadastrado para **múltiplos CPFs** na planilha:
o primeiro CPF é entregue e os seguintes levam 131049.

**O que falta investigar:** dos 480 telefones com 131049, quantos são casos de
CPF-compartilhado (telefone já recebeu o HSM para outro CPF) versus casos em que
o número foi bloqueado pela Meta por volume geral de mensagens recebidas?
Para isso, cruzar os 480 telefones com erro 131049 contra os telefones com
`envio_meta_datahora IS NOT NULL` na mesma campanha e verificar a sobreposição.

In [22]:
query_funil_envio = f"""
WITH
seed_falha AS (
    {SEED_FALHA_SQL}
),

hsm AS (
    SELECT
        CASE
            WHEN hsm.nome_hsm = 'recado_dia_do_servidor_v3'   THEN 'Dia Servidor'
        END AS campanha,
        hsm.id_disparo,
        contato.contato_telefone,
        contato.cpf,
        hsm.indicador_falha,
        hsm.descricao_falha,
        hsm.indicador_falha_meta,
        hsm.falha_meta_descricao,
        hsm.indicador_falha_plataforma,
        hsm.falha_plataforma_descricao,
        hsm.entrega_datahora,
        hsm.envio_datahora,
        hsm.leitura_datahora,
        entrega_observada_indicador,
        leitura_observada_indicador,
        foi_respondida
    FROM {FCT_CHATBOT_V2}
    WHERE fonte = 'HSM'
      AND hsm.nome_hsm IN ('recado_dia_do_servidor_v3')
      AND data_particao >= "2026-10-28"
      and mensagem_datahora >= '2026-10-28T10:02:02.109000'
),

-- join com o seed para classificar cada falha por origem
hsm_com_origem AS (
    SELECT
        h.*,
        sf.origem_tipo
    FROM hsm h
    LEFT JOIN seed_falha sf
        ON h.indicador_falha IS TRUE
        AND h.descricao_falha LIKE CONCAT('%', sf.codigo_sigla, '%')
)

SELECT
    campanha,
    COUNT(DISTINCT id_disparo)                                              AS total_disparos,
    COUNT(DISTINCT contato_telefone)                                              AS total_numeros_disparados_dist,
    COUNT(DISTINCT cpf)                                                           AS total_cpfs_disparados_dist,
    -- erros por origem
    COUNT(DISTINCT
        CASE WHEN indicador_falha_plataforma
             THEN contato_telefone END)                                           AS erro_salesforce,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta
             THEN contato_telefone END)                                           AS erro_meta,
    ROUND(100*COUNT(DISTINCT
            CASE WHEN indicador_falha_meta or indicador_falha_plataforma
                 THEN contato_telefone END)/COUNT(DISTINCT id_disparo),1)         AS pct_falha_geral,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta 
            and (falha_meta_descricao not like "%experiment%" and falha_meta_descricao not like "%healthy%")
                THEN contato_telefone END)                                        AS erro_meta_numero,
    ROUND(100*COUNT(DISTINCT
        CASE WHEN (indicador_falha_meta or indicador_falha_plataforma)
                and (falha_meta_descricao not like "%experiment%" and falha_meta_descricao not like "%healthy%")
                THEN contato_telefone END)/COUNT(DISTINCT id_disparo),1)         AS pct_falha_numero,
    -- ciclo de entrega (eventos observados, sem inferência por resposta)
    COUNT(DISTINCT
            CASE WHEN envio_datahora IS NOT NULL or entrega_datahora IS NOT NULL 
                    or leitura_datahora IS NOT NULL
                 THEN contato_telefone END)                                         AS enviados,
    COUNT(DISTINCT
            CASE WHEN entrega_datahora IS NOT NULL or leitura_datahora IS NOT NULL
                 THEN contato_telefone END)                                         AS entregues,
    COUNT(DISTINCT
        CASE WHEN entrega_observada_indicador IS TRUE
             THEN contato_telefone END)                                           AS recebidos,
    COUNT(DISTINCT
        CASE WHEN leitura_observada_indicador IS TRUE
             THEN contato_telefone END)                                           AS lidos,
    COUNT(DISTINCT
        CASE WHEN foi_respondida IS TRUE
             THEN contato_telefone END)                                           AS respondidos,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta 
            and (falha_meta_descricao like "%experiment%")
                THEN contato_telefone END)                                           AS erro_meta_experiment,
    COUNT(DISTINCT
        CASE WHEN indicador_falha_meta 
            and (falha_meta_descricao like "%healthy%")
                THEN contato_telefone END)                                           AS erro_meta_healthy,
FROM hsm_com_origem
GROUP BY campanha
ORDER BY campanha
"""

df_funil_envio_ = run_query(query_funil_envio)
print("=== Funil de envio (fct_chatbot_v2) ===")
df_funil_envio_

=== Funil de envio (fct_chatbot_v2) ===


,campanha,total_disparos,total_numeros_disparados_dist,total_cpfs_disparados_dist,erro_salesforce,erro_meta,pct_falha_geral,erro_meta_numero,pct_falha_numero,enviados,entregues,recebidos,lidos,respondidos,erro_meta_experiment,erro_meta_healthy
0,Dia Servidor,117526,117090,117524,0,15870,13.5,11956,10.2,101233,99947,99947,63398,457,2124,1790
